# 🧬 Project 5 — Semantic Correspondence (COMPLETE PIPELINE)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data
# Per PF-Pascal, assicurati che la cartella sia in ./data/PF-Pascal
!mkdir -p ./data/PF-Pascal

import os, torch, gc
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print('[INFO] GPU Cleared.')

## 🔍 1. Evaluation Baseline (Multibackbone)

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt

In [ ]:
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --batch_size 1 --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

## 🚀 2. Training (LoRA, Curriculum)

In [ ]:
CKPT_PATH = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
if not os.path.exists(CKPT_PATH):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir "$DRIVE_CKPTS/lora_only"
else: print(f'[OK] Checkpoint già presente.')

## 🎯 3. Raffinamento (Adaptive Window Ablation)

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
print('Valutazione CON Adaptive Window...')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$CKPT_LORA" --results_file /content/drive/MyDrive/AML/Results/lora_with_aw.txt
print('Valutazione SENZA Adaptive Window...')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$CKPT_LORA" --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/lora_no_aw.txt

## 🌍 4. Generalizzazione e Robustezza

In [ ]:
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
# Test PF-Pascal
!python evaluate.py --dataset_root ./data/PF-Pascal --dataset_type pfpascal --checkpoint "$CKPT_LORA" --results_file /content/drive/MyDrive/AML/Results/gen_pfpascal.txt

In [ ]:
# Test Robustezza Geometrica
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$CKPT_LORA" --results_file /content/drive/MyDrive/AML/Results/robustness.txt

## ⚖️ 5. Showcase Finali

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')